# DREF Data Prep And Metric Contract

This notebook reads the source workbook, cleans the `ALL_DATA` sheet, applies the base DREF-family scope, and prepares reusable datasets for the implementation-phase figures.

Outputs are written to `data/processed/` so the slide notebooks can stay focused on figure construction.

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 200)
pd.set_option('display.float_format', lambda value: f'{value:,.2f}')

In [5]:
ROOT = Path.cwd().resolve()
# Notebooks live in <project_root>/notebooks/ — navigate up to project root
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

WORKBOOK_PATH = ROOT / 'DREF_MasterDataset_v1.1 .xlsx'
PROCESSED_DIR = ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

Q1_2026_CUTOFF = pd.Timestamp('2026-03-31')
BASE_APPEAL_TYPES = ['DREF', 'i-DREF', 'a-DREF']
CONTEXT_APPEAL_TYPES = ['EA', 'EAP', 's-EAP']

print('Workbook:', WORKBOOK_PATH)
print('Workbook exists:', WORKBOOK_PATH.exists())
print('Processed dir:', PROCESSED_DIR)


Workbook: C:\Users\arun.gandhi\Downloads\DREF_GA_visualizations\DREF_MasterDataset_v1.1 .xlsx
Workbook exists: True
Processed dir: C:\Users\arun.gandhi\Downloads\DREF_GA_visualizations\data\processed


In [6]:
all_data_raw = pd.read_excel(WORKBOOK_PATH, sheet_name='ALL_DATA')
oda_sheet = pd.read_excel(WORKBOOK_PATH, sheet_name='ODA Countries')

print('ALL_DATA shape:', all_data_raw.shape)
print('ODA sheet shape:', oda_sheet.shape)

ALL_DATA shape: (2506, 78)
ODA sheet shape: (142, 11)


c:\Users\arun.gandhi\Downloads\DREF_GA_visualizations\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


In [7]:
def slugify_column(name: str) -> str:
    text = re.sub(r'[^0-9a-zA-Z]+', '_', str(name).strip().lower())
    text = re.sub(r'_+', '_', text).strip('_')
    return text

all_data = all_data_raw.rename(columns={column: slugify_column(column) for column in all_data_raw.columns}).copy()

text_columns = [
    'appeal_id', 'appeal_type', 'pillar', 'allocation_type', 'country', 'region',
    'weather_vs_non_weather', 'natural_vs_non_natural', 'disaster_definition',
    'disaster_name', 'crisis_categorization', 'type_of_onset', 'silent_cancelled'
]
for column in text_columns:
    if column in all_data.columns:
        all_data[column] = all_data[column].fillna('').astype(str).str.strip()

date_columns = [
    'date_of_disaster_trigger_date',
    'date_of_appeal_request_from_ns',
    'date_of_approval_enc_start_date',
    'final_approval_quality',
    'date_of_publication_dref_eap_summary',
    'end_date_of_operation',
    'final_report_due_date',
    'final_report_received_date'
]
for column in date_columns:
    if column in all_data.columns:
        all_data[column] = pd.to_datetime(all_data[column], errors='coerce')

numeric_columns = [
    'total_approved_chf', 'targeted_people', 'affected_people', 'beneficiaries_asssisted',
    'average_cost_per_person', 'average_time_disaster_to_approval', 'days_in_hq',
    'disaster_to_quality_approval', 'eap_lead_time_days', 'cash_advance_from_checklist',
    'fast_track_25', 'amount_transferred'
]
for column in numeric_columns:
    if column in all_data.columns:
        all_data[column] = pd.to_numeric(all_data[column], errors='coerce')

all_data['year'] = pd.to_numeric(all_data['year'], errors='coerce').astype('Int64')
all_data['approval_date'] = all_data['date_of_approval_enc_start_date']
all_data['approval_year'] = all_data['approval_date'].dt.year.astype('Int64')
all_data['approval_month'] = all_data['approval_date'].dt.month.astype('Int64')
all_data['is_silent_or_cancelled'] = all_data['silent_cancelled'].str.lower().isin(['yes', 'silent', 'cancelled'])

all_data.head()

,index,appeal_id,code_number,child_id,appeal_s_allocation,pillar,appeal_type,allocation_type,country,region_code,region,weather_vs_non_weather,natural_vs_non_natural,disaster_definition,disaster_name,silent_cancelled,elegible_for_insurance,type_of_onset,crisis_categorization,total_approved_chf,cash_advance_from_checklist,fast_track_25,readiness_costs,prepositioned_stocks,early_action_costs,eap_status,eap_lead_time_days,date_of_disaster_trigger_date,date_of_appeal_request_from_ns,date_of_c_a_signature,date_of_appeal_request_from_regions,review_start_date,date_of_approval_enc_start_date,final_approval_quality,disaster_to_quality_approval,average_time_disaster_to_approval,kpi,days_in_hq,date_of_approval_2nd_allocation,date_of_publication_dref_eap_summary,date_of_publication_emergency_appeal,end_date_of_operation,operation_timeframe,final_report_due_date,final_report_received_date,latest_report,status_of_final_report,report_delayed,report_delayed_number_of_days,affected_people,targeted_people,average_cost_per_person,operational_costs,beneficiaries_asssisted,female_assisted,male_asissted,timeframe_extension,additional_timeframe_extention,project_code,number_of_approvers,number_of_reviewers,date_of_code_open,enc_date_of_e_contract_initiation,enc_date_of_signature,date_of_e_contract_initiation,final_date_of_signature,time_for_agreement_signature,contract_status,amount_transferred,canada_gvt,echo,nlrc,belgian_gvt,other_donor_chf,donor_pns,reimbursed_for_finance,comments,year,approval_date,approval_year,approval_month,is_silent_or_cancelled
0,1,MDRCD006,006,NaN,NaN,Anticipatory,i-DREF,Grant,Democratic Republic of the Congo,AF,Africa,Non-weather-related,Non-natural,Epidemic,Ebola,,NaN,Imminent,,146404,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2008-12-26,NaT,NaT,2009-01-07,NaT,2009-01-12,NaT,NaN,17.00,NaN,5.00,NaT,NaT,NaT,NaT,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,"15,000.00",9.76,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"67,659.00",NaN,NaN,NaN,"1,083.00",NaN,2009,2009-01-12,2009,1,False
1,2,MDRTG002,002,NaN,NaN,Response,DREF,Grant,Togo,AF,Africa,Non-weather-related,Non-natural,Epidemic,Cholera,,NaN,Sudden,,75376,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2008-12-15,NaT,NaT,2009-01-13,NaT,2009-01-18,NaT,NaN,34.00,NaN,5.00,NaT,NaT,NaT,NaT,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,"353,120.00",0.21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"14,961.00",NaN,NaN,NaN,"4,257.00",NaN,2009,2009-01-18,2009,1,False
2,3,MDRCF003,003,NaN,NaN,Response,DREF,Grant,Central African Republic,AF,Africa,Non-weather-related,Non-natural,Epidemic,Yellow fever,,NaN,Slow,,30100,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2009-01-05,NaT,NaT,2009-01-13,NaT,2009-01-18,NaT,NaN,13.00,NaN,5.00,NaT,NaT,NaT,NaT,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,"258,492.00",0.12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"14,852.00",NaN,NaN,NaN,"1,638.00",NaN,2009,2009-01-18,2009,1,False
3,4,MDRMW004,004,NaN,NaN,Response,DREF,Grant,Malawi,AF,Africa,Weather-related,Natural,Flood,Floods and cholera,,NaN,Sudden,,71022,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2008-08-11,NaT,NaT,2009-01-16,NaT,2009-01-21,NaT,NaN,163.00,NaN,5.00,NaT,NaT,NaT,NaT,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,"16,380.00",4.34,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"37,130.00",NaN,NaN,NaN,894.00,NaN,2009,2009-01-21,2009,1,False
4,5,MDRZM005,005,NaN,NaN,Response,DREF,Grant,Zambia,AF,Africa,Non-weather-related,Non-natural,Epidemic,Cholera,,NaN,Sudden,,60960,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2009-01-10,NaT,NaT,2009-01-22,NaT,2009-01-27,NaT,NaN,17.00,NaN,5.00,NaT,NaT,NaT,NaT,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,"31,100.00",1.96,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"37,560.00",NaN,NaN,NaN,"2,342.00",NaN,2009,2009-01-27,2009,1,False


In [ ]:
# Exclude silent/cancelled records from the base analytical scope (plan data contract requirement)
base_dref = all_data[
    all_data['appeal_type'].isin(BASE_APPEAL_TYPES)
    & ~all_data['is_silent_or_cancelled']
].copy()
context_records = all_data[all_data['appeal_type'].isin(CONTEXT_APPEAL_TYPES)].copy()

recent_base = base_dref[base_dref['approval_year'].between(2022, 2026)].copy()
q1_history = recent_base[recent_base['approval_month'].between(1, 3)].copy()
q1_2025 = q1_history[q1_history['approval_year'] == 2025].copy()
q1_2026 = q1_history[(q1_history['approval_year'] == 2026) & (q1_history['approval_date'] <= Q1_2026_CUTOFF)].copy()
protocols_q1_2026 = context_records[
    context_records['appeal_type'].isin(['EAP', 's-EAP'])
    & (context_records['approval_date'].dt.year == 2026)
    & (context_records['approval_date'].dt.month <= 3)
].copy()

oda_countries = (
    oda_sheet.iloc[:, 0]
    .dropna()
    .astype(str)
    .str.strip()
    .replace({'ODA Country': np.nan})
    .dropna()
    .drop_duplicates()
    .sort_values()
)

q1_history.to_csv(PROCESSED_DIR / 'q1_history_base_dref.csv', index=False)
q1_2025.to_csv(PROCESSED_DIR / 'q1_2025_base_dref.csv', index=False)
q1_2026.to_csv(PROCESSED_DIR / 'q1_2026_base_dref.csv', index=False)
protocols_q1_2026.to_csv(PROCESSED_DIR / 'q1_2026_protocol_context.csv', index=False)
oda_countries.to_frame(name='country').to_csv(PROCESSED_DIR / 'oda_countries_reference.csv', index=False)

silent_count = all_data[all_data['appeal_type'].isin(BASE_APPEAL_TYPES) & all_data['is_silent_or_cancelled']].shape[0]
print(f'Silent/cancelled excluded from base_dref: {silent_count}')
print('Base DREF rows (active only):', len(base_dref))
print('Q1 history rows:', len(q1_history))
print('Q1 2025 rows:', len(q1_2025))
print('Q1 2026 rows:', len(q1_2026))
print('Q1 2026 protocol context rows:', len(protocols_q1_2026))
print('ODA reference countries:', len(oda_countries))


Base DREF rows: 1841
Q1 history rows: 180
Q1 2025 rows: 33
Q1 2026 rows: 54
Q1 2026 protocol context rows: 8
ODA reference countries: 141


In [ ]:
metric_contract = pd.DataFrame([
    {
        'figure_group': 'slides_2_10',
        'metric': 'base_scope',
        'definition': 'Appeal Type in DREF, i-DREF, a-DREF unless slide-specific exception is documented',
        'dataset': 'base_dref'
    },
    {
        'figure_group': 'slides_2_10',
        'metric': 'silent_cancelled_exclusion',
        'definition': 'Records where silent_cancelled field is yes/silent/cancelled are excluded from base_dref and all derived datasets',
        'dataset': 'base_dref'
    },
    {
        'figure_group': 'slides_2_10',
        'metric': 'q1_period',
        'definition': 'Approval month in January to March; 2026 also capped at 2026-03-31',
        'dataset': 'q1_history / q1_2026'
    },
    {
        'figure_group': 'slide_2',
        'metric': 'triggered_protocols_context',
        'definition': 'EAP and s-EAP records in Q1 2026 used as contextual layer only',
        'dataset': 'protocols_q1_2026'
    },
    {
        'figure_group': 'slide_7',
        'metric': 'non_oda',
        'definition': 'Country not found in the ODA Countries reference sheet',
        'dataset': 'oda_countries_reference'
    },
    {
        'figure_group': 'slide_5',
        'metric': 'localization',
        'definition': 'No explicit localization field found in ALL_DATA; current figure build requires manual values or a confirmed formula',
        'dataset': 'manual_pending'
    }
])

metric_contract.to_csv(PROCESSED_DIR / 'metric_contract.csv', index=False)
display(metric_contract)


,figure_group,metric,definition,dataset
0,slides_2_10,base_scope,"Appeal Type in DREF, i-DREF, a-DREF unless sli...",base_dref
1,slides_2_10,q1_period,Approval month in January to March; 2026 also ...,q1_history / q1_2026
2,slide_2,triggered_protocols_context,EAP and s-EAP records in Q1 2026 used as conte...,protocols_q1_2026
3,slide_7,non_oda,Country not found in the ODA Countries referen...,oda_countries_reference
4,slide_5,localization,No explicit localization field found in ALL_DA...,manual_pending


In [10]:
summary = pd.DataFrame({
    'dataset': ['base_dref', 'q1_history', 'q1_2025', 'q1_2026', 'protocols_q1_2026'],
    'rows': [len(base_dref), len(q1_history), len(q1_2025), len(q1_2026), len(protocols_q1_2026)],
    'min_approval_date': [
        base_dref['approval_date'].min(),
        q1_history['approval_date'].min(),
        q1_2025['approval_date'].min(),
        q1_2026['approval_date'].min(),
        protocols_q1_2026['approval_date'].min()
    ],
    'max_approval_date': [
        base_dref['approval_date'].max(),
        q1_history['approval_date'].max(),
        q1_2025['approval_date'].max(),
        q1_2026['approval_date'].max(),
        protocols_q1_2026['approval_date'].max()
    ]
})

display(summary)
display(q1_2026[['appeal_id', 'country', 'pillar', 'disaster_definition', 'total_approved_chf']].head(10))

,dataset,rows,min_approval_date,max_approval_date
0,base_dref,1841,2009-01-12,2026-04-06
1,q1_history,180,2022-01-04,2026-03-31
2,q1_2025,33,2025-01-08,2025-03-29
3,q1_2026,54,2026-01-08,2026-03-31
4,protocols_q1_2026,8,2026-01-14,2026-03-25


,appeal_id,country,pillar,disaster_definition,total_approved_chf
2433,MDRGM017,Gambia,Response,Population Movement,235260
2434,MDRCO034,Colombia,Anticipatory,Population Movement,90800
2436,MDRTZ043,Tanzania,Response,Flood,185455
2437,MDRMG026,Madagascar,Response,Epidemic,394995
2438,MDRXK002,Kosovo,Response,Flood,157148
2439,MDRRS016,Serbia,Response,Other,342374
2440,MDRZA022,South Africa,Response,Flood,498950
2442,MDRKM014,Comoros,Response,Epidemic,428215
2443,MDRMK011,North Macedonia,Response,Flood,150613
2444,MDRGT027,Guatemala,Response,Other,330155


## Data Caveats For The Implementation Phase

- Slide 4 uses `crisis_categorization`, which is sparsely populated in the raw data. The figure notebook builds a provisional version from available rows only.
- Slide 5 localization values do not appear as explicit fields in `ALL_DATA`. The slide notebook uses the published slide values until a reproducible workbook formula is confirmed.
- Slide 7 non-ODA logic is derived from the `ODA Countries` reference sheet by exclusion. If the workbook uses a more specific rule, update the reference logic before final production.
- Slide 10 intentionally allows `EA` context because the story mixes Emergency Appeal loans with imminent DREF approvals.